# D1-03 Manual LCA with NumPy matrices (Heijungs and Suh 2002) vs Brightway

This notebook reconstructs the matrix-based computation logic behind LCA by hand.

We do **not** call `brightway` directly here. Instead, we manually reproduce the same computational structure so that the transition to `D1-04` is clear.


## Learning goals

- Use standard LCA matrix notation to define a small product system.
- Solve for the scaling vector from a demand vector.
- Compute inventory and impact results manually with `numpy`.
- Understand why preserving matrix structure is useful for contribution analysis later.
- Build the conceptual bridge to the way `brightway` performs LCA calculations.


## Background references

- Heijungs, R., & Suh, S. (2002). *The computational structure of life cycle assessment*. Kluwer Academic Publishers. https://doi.org/10.1007/978-94-015-9900-9
- Mutel, C. (2017). *Brightway: An open source framework for life cycle assessment*. Journal of Open Source Software, 2(12), 236. https://doi.org/10.21105/joss.00236


## 1) Notation used in this notebook

We use the following notation:

- `A`: technosphere matrix
- `B`: intervention matrix
- `f`: demand vector
- `s`: scaling vector, obtained from `A s = f`
- `G = B diag(s)`: inventory results kept in matrix form
- `g = G 1 = B s`: aggregated inventory vector
- `Q`: diagonalized vector of characterization factors
- `H = Q G`: characterized inventory matrix
- `h = H 1`: aggregated characterized result

The key bridge to `brightway` is that preserving `G` and `H` in matrix form keeps the contribution of each activity visible.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

pd.options.display.float_format = '{:,.3f}'.format


## 2) Example 1: the smallest possible system

Start with one activity, one elementary flow, and one impact category.
This is only to make the symbols concrete before moving to larger systems.


In [ ]:
A1 = np.array([[1.0]])
B1 = np.array([[2.0]])
f1 = np.array([1.0])
q1 = np.array([39.5])
Q1 = np.diag(q1)

s1 = np.linalg.solve(A1, f1)
G1 = B1 @ np.diag(s1)
g1 = B1 @ s1
H1 = Q1 @ G1
h1 = H1.sum()

print('Scaling vector s1:', s1)
print('Inventory vector g1:', g1)
print('Characterized score h1:', h1)


## 3) Example 2: a two-activity abstract system

Now we move to an abstract system with two products and two elementary flows.
The names stay generic so that the mathematical structure stays central.


In [ ]:
products = ['Product 1', 'Product 2']
activities = ['Activity 1', 'Activity 2']
flows = ['Flow X', 'Flow Y']

A2 = np.array([
    [1.0,  0.0],
    [-0.3, 1.0],
])
B2 = np.array([
    [1.5, 0.2],
    [0.1, 0.8],
])
f2 = np.array([1.0, 0.0])
q2 = np.array([1.0, 5.0])
Q2 = np.diag(q2)

s2 = np.linalg.solve(A2, f2)
G2 = B2 @ np.diag(s2)
g2 = B2 @ s2
H2 = Q2 @ G2
h2 = H2.sum()

print('Technosphere matrix A2')
display(pd.DataFrame(A2, index=products, columns=activities))

print('Intervention matrix B2')
display(pd.DataFrame(B2, index=flows, columns=activities))

print('Demand vector f2')
display(pd.Series(f2, index=products, name='demand'))

print('Scaling vector s2')
display(pd.Series(s2, index=activities, name='scale'))

print('Inventory matrix G2 = B2 @ diag(s2)')
display(pd.DataFrame(G2, index=flows, columns=activities))

print('Characterized inventory matrix H2 = Q2 @ G2')
display(pd.DataFrame(H2, index=flows, columns=activities))

print('Aggregated inventory g2:', g2)
print('Aggregated characterized score h2:', h2)


## Checkpoint 1

Change the functional unit so that demand is `2` units of `Product 1`.
Recompute `s`, `G`, and the total characterized score.


In [ ]:
# TODO
# f2_new = ...
# s2_new = ...
# G2_new = ...
# h2_new = ...


In [ ]:
f2_new = np.array([2.0, 0.0])
s2_new = np.linalg.solve(A2, f2_new)
G2_new = B2 @ np.diag(s2_new)
H2_new = Q2 @ G2_new
h2_new = H2_new.sum()

print('New scaling vector:', s2_new)
display(pd.DataFrame(G2_new, index=flows, columns=activities))
print('New total characterized score:', h2_new)


## 4) Example 3: a compact regionalized lithium supply chain

This example is adapted from the supplementary numerical note you referenced.
It stays small enough to calculate by hand, but it already contains locations and region-specific water withdrawals.

Processes:
- Li+ extraction (CL)
- Hydropower generation (CL)
- Li2CO3 production (CN)
- Hydropower generation (CN)
- CAM production (DE)
- Hydropower generation (DE)


In [ ]:
products3 = [
    'Li+ (aq.) [kg]',
    'Electricity (CL) [kWh]',
    'Li2CO3 [kg]',
    'Electricity (CN) [kWh]',
    'CAM [kg]',
    'Electricity (DE) [kWh]',
]
activities3 = [
    'Li+ extraction (CL)',
    'Hydropower (CL)',
    'Li2CO3 production (CN)',
    'Hydropower (CN)',
    'CAM production (DE)',
    'Hydropower (DE)',
]
flows3 = ['Water, underground [kg]', 'Water, surface [kg]', 'Water, unspecified [kg]']

A3 = np.array([
    [ 1.0,  0.0, -0.8, 0.0, -0.1, 0.0],
    [-0.5,  1.0,  0.0, 0.0,-25.0, 0.0],
    [ 0.0,  0.0,  1.0, 0.0, -0.2, 0.0],
    [ 0.0,  0.0, -3.0, 1.0,  0.0, 0.0],
    [ 0.0,  0.0,  0.0, 0.0,  1.0, 0.0],
    [ 0.0,  0.0,  0.0, 0.0,-25.0, 1.0],
])
B3 = np.array([
    [2.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 3.0, 0.0, 0.0, 0.0],
    [0.0, 0.02,0.0,0.015,0.2,0.005],
])
f3 = np.array([0.0, 0.0, 0.0, 0.0, 1.0, 0.0])
q3 = np.array([39.5, 39.5, 39.5])
Q3 = np.diag(q3)

s3 = np.linalg.solve(A3, f3)
G3 = B3 @ np.diag(s3)
g3 = B3 @ s3
H3 = Q3 @ G3
h3 = H3.sum()

print('Technosphere matrix A3')
display(pd.DataFrame(A3, index=products3, columns=activities3))

print('Intervention matrix B3')
display(pd.DataFrame(B3, index=flows3, columns=activities3))

print('Demand vector f3')
display(pd.Series(f3, index=products3, name='demand'))

print('Scaling vector s3')
display(pd.Series(s3, index=activities3, name='scale'))

print('Inventory matrix G3 = B3 @ diag(s3)')
display(pd.DataFrame(G3, index=flows3, columns=activities3))

print('Characterized inventory matrix H3 = Q3 @ G3')
display(pd.DataFrame(H3, index=flows3, columns=activities3))

print('Aggregated inventory g3')
display(pd.Series(g3, index=flows3, name='inventory'))

print('Total characterized score h3:', h3)


## Why keep `G` and `H` in matrix form?

If we collapse directly to a single total, we lose the activity-by-activity structure.
Keeping `G` and `H` in matrix form makes it easy to see which activity contributes to which flow and which characterized result.

That is the main conceptual bridge to the contribution analysis work later in Day 1.


## Checkpoint 2

Using `H3`, identify the process with the largest contribution to the total characterized result.
Then identify which elementary flow dominates the total score.


In [ ]:
# TODO
# column_contrib = ...
# row_contrib = ...


In [ ]:
column_contrib = pd.DataFrame(H3, index=flows3, columns=activities3).sum(axis=0).sort_values(ascending=False)
row_contrib = pd.DataFrame(H3, index=flows3, columns=activities3).sum(axis=1).sort_values(ascending=False)

print('Activity contributions to the total characterized score:')
display(column_contrib)

print('Elementary-flow contributions to the total characterized score:')
display(row_contrib)

print('Largest contributing activity:', column_contrib.index[0])
print('Dominant elementary flow:', row_contrib.index[0])


## 5) Bridge to `brightway`

The manual workflow above is the conceptual foundation for `brightway` calculations:

- activities and exchanges define `A` and `B`
- the demand dictionary defines `f`
- solving the system gives the scaling vector
- characterization factors define `Q`
- preserving matrix structure supports contribution analysis

In `D1-04`, we stop constructing these objects by hand and let `brightway` build them from imported data.


## Recap

After this notebook, you should now be able to:

- write down the main LCA matrices and vectors
- solve `A s = f` with `numpy`
- compute inventory and characterized results manually
- explain why preserving matrix form is useful
- connect the manual matrix workflow to the software workflow in `brightway`
